# STEP 1. 재난·안전 RAG 지식베이스 구축
셀 순서대로 실행하세요. 셀2(doc변환), 셀8(bge-m3 다운로드)이 오래 걸립니다.

In [23]:
# [셀1] 경로 설정 + 파일 목록 확인
from pathlib import Path
from collections import Counter

RAW_DIR   = Path(r"D:\충북대\지능화캡스톤\data\raw")
INDEX_DIR = Path(r"D:\rag_index")
INDEX_DIR.mkdir(parents=True, exist_ok=True)

assert RAW_DIR.exists(), f"경로 없음: {RAW_DIR}"

all_files = [f for f in RAW_DIR.rglob("*") if f.is_file()]
ext_counter = Counter(f.suffix.lower() for f in all_files)
print(f"총 파일 수: {len(all_files)}개")
for ext, cnt in sorted(ext_counter.items()):
    print(f"  {ext or '(없음)'}: {cnt}개")

총 파일 수: 53개
  .doc: 8개
  .docx: 1개
  .hwpx: 1개
  .jpg: 8개
  .pdf: 30개
  .png: 5개


In [24]:
# [셀2] .doc → PDF 자동 변환 (Word 설치 필요)
from docx2pdf import convert

doc_files = list(RAW_DIR.rglob("*.doc"))
print(f"변환할 .doc 파일: {len(doc_files)}개")

for doc_path in doc_files:
    pdf_path = doc_path.with_suffix(".pdf")
    if pdf_path.exists():
        print(f"  [SKIP] {pdf_path.name}")
        continue
    try:
        convert(str(doc_path), str(pdf_path))
        print(f"  [OK]   {doc_path.name} -> {pdf_path.name}")
    except Exception as e:
        print(f"  [FAIL] {doc_path.name}: {e}")

print("완료")

변환할 .doc 파일: 8개
  [FAIL] 강풍 국민행동요령 매뉴얼.doc: 
  [FAIL] 대설 국민행동요령 매뉴얼.doc: 
  [FAIL] 강풍 국민행동요령 매뉴얼.doc: 
  [FAIL] 태풍 국민행동요령 매뉴얼_2.doc: 
  [FAIL] 폭염 국민행동요령 매뉴얼.doc: 
  [FAIL] 홍수 국민행동요령 매뉴얼.doc: 
  [FAIL] 홍수 국민행동요령 매뉴얼.doc: 
  [FAIL] 황사 국민행동요령 매뉴얼.doc: 
완료


In [25]:
# [셀3] EasyOCR 초기화 (첫 실행시 모델 다운로드 1~2분)
import easyocr
print("EasyOCR 초기화 중...")
ocr_reader = easyocr.Reader(['ko', 'en'], gpu=True)
print("완료")

Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.


EasyOCR 초기화 중...
완료


In [26]:
# [셀4] 텍스트 추출 함수 정의
import fitz
import docx
import numpy as np
from PIL import Image


def extract_pdf(path):
    """텍스트 PDF → get_text(), 이미지 PDF → OCR"""
    text = ""
    try:
        with fitz.open(str(path)) as doc:
            for page in doc:
                page_text = page.get_text().strip()
                if len(page_text) > 30:
                    text += page_text + "\n"
                else:
                    pix = page.get_pixmap(matrix=fitz.Matrix(2, 2))
                    arr = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.h, pix.w, pix.n)
                    if pix.n == 4:
                        arr = arr[:, :, :3]
                    res = ocr_reader.readtext(arr, detail=0, paragraph=True)
                    text += " ".join(res) + "\n"
    except Exception as e:
        print(f"  [WARN] PDF 오류 {path.name}: {e}")
    return text.strip()


def extract_docx(path):
    try:
        d = docx.Document(str(path))
        return "\n".join(p.text for p in d.paragraphs if p.text.strip())
    except Exception as e:
        print(f"  [WARN] DOCX 오류 {path.name}: {e}")
        return ""


def extract_image(path):
    """PIL로 로드 후 OCR (한글 경로 문제 방지)"""
    try:
        img = Image.open(str(path)).convert("RGB")
        arr = np.array(img)
        res = ocr_reader.readtext(arr, detail=0, paragraph=True)
        return " ".join(res)
    except Exception as e:
        print(f"  [WARN] OCR 오류 {path.name}: {e}")
        return ""


def get_disaster_type(path):
    folder = path.parent.name
    return folder.split("_")[0] if "_" in folder else folder


print("함수 정의 완료")

함수 정의 완료


In [27]:
# [셀5] 전체 파일 로드
from tqdm import tqdm
from langchain_core.documents import Document

SUPPORT_EXT = {".pdf", ".docx", ".jpg", ".jpeg", ".png"}

target_files = [f for f in RAW_DIR.rglob("*")
                if f.is_file() and f.suffix.lower() in SUPPORT_EXT]
print(f"처리할 파일 수: {len(target_files)}개")

raw_docs, fail_list = [], []

for fpath in tqdm(target_files, desc="파일 로드"):
    ext = fpath.suffix.lower()
    if ext == ".pdf":
        text = extract_pdf(fpath)
    elif ext == ".docx":
        text = extract_docx(fpath)
    else:
        text = extract_image(fpath)

    if len(text) < 50:
        fail_list.append(fpath.name)
        continue

    raw_docs.append(Document(
        page_content=text,
        metadata={
            "source_file": fpath.name,
            "disaster_type": get_disaster_type(fpath),
            "file_type": ext,
        }
    ))

print(f"\n로드 성공: {len(raw_docs)}개")
if fail_list:
    print(f"실패 ({len(fail_list)}개): {fail_list}")

처리할 파일 수: 44개


파일 로드:   0%|          | 0/44 [00:00<?, ?it/s]d:\rag_env\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
파일 로드: 100%|██████████| 44/44 [20:49<00:00, 28.40s/it] 


로드 성공: 44개


In [28]:
# [셀6] 청크 분할
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=80,
    separators=["\n\n", "\n", ". ", " ", ""],
)
chunks = splitter.split_documents(raw_docs)
print(f"총 청크 수: {len(chunks)}개")

type_counts = Counter(c.metadata["disaster_type"] for c in chunks)
print("\n재난 유형별 청크 수:")
for dtype, cnt in sorted(type_counts.items(), key=lambda x: -x[1]):
    print(f"  {dtype}: {cnt}개")

총 청크 수: 305개

재난 유형별 청크 수:
  지진: 92개
  황사: 61개
  화재: 53개
  화학사고재난: 27개
  미세먼지: 24개
  감염병예방: 12개
  강풍: 10개
  산불: 10개
  한파: 5개
  대설: 4개
  태풍: 2개
  호우: 2개
  홍수: 2개
  풍랑: 1개


In [29]:
# [셀7] 샘플 확인 (텍스트 품질 눈으로 체크)
for i, chunk in enumerate(chunks[:5]):
    print(f"\n--- {i+1} | {chunk.metadata['disaster_type']} | {chunk.metadata['source_file']} ---")
    print(chunk.page_content[:250])


--- 1 | 감염병예방 | 감염병 예방을 위한 손씻기 6단계.pdf ---
올바른 손씻기 
방법 6단계
화장실 이용 후
음식을 먹기 전 · 후
음식을 준비할 때
(생고기, 가금류 등 접촉 후)
아픈 사람을 간병할 때
기저귀를 갈거나, 화장실  
다녀온 아이를 닦아준 후
베인 상처나 
창상을 다룰 때
코를 풀거나  
기침, 재채기 후
쓰레기를 취급한 후
반려동물, 사료,  
관련 폐기물 등에 접촉한 후
반려동물에 접촉하거나  
먹이를 준 후
올바른 손씻기가
필요한 순간
*출처:미국 질병통제예방센터(Centers for 

--- 2 | 감염병예방 | 감염병 예방을 위한 손씻기 6단계.pdf ---
비비기
두 손 모아
두 손 모아
4
손톱 밑을 손바닥에  
문지르며 마무리
손톱 밑
손톱 밑
6
손바닥을 마주대고  
문지르기
손바닥
손바닥
1
손깍지를 끼고  
손가락 사이 닦아주기
손가락 사이
손가락 사이
3
엄지손가락을 돌려주며  
닦아주기
엄지손가락
엄지손가락
5
올바른 손씻기 6단계 
실천 매뉴얼
23.10.10.
호흡기질환
20% 
발생 예방
설사질환
30% 
발생 예방
*출처:미국 질병통제예방센터(Centers for Diseas

--- 3 | 감염병예방 | 감염병 예방을 위한 손씻기 6단계.pdf ---
주요 장관감염증
올바른 손씻기로 예방 가능한
주요 호흡기감염증
*출처 : 2022년 수인성 및 식품매개 감염병 관리지침, 질병관리청, 2022
*출처 : 2021-2022년 인플루엔자관리지침 및 2023년 법정 진단·신고 기준
A형간염 환자의 대변에 의해 오염된 음식과  
물을 통해 전파 가능 (잠복기 15~50일)
전파경로
발열, 피로, 식욕부진, 복통, 구역 및 구토, 황달 등
증상
· A형간염 예방접종 2회 모두 받기
· 반드시 끓인 물,

--- 4 | 감염병예방 | 감염병 예방을 위한 손씻기 6단계.pdf ---
(잠복기 12시간~7일)
전파경로
고열, 구토, 복통, 설사 등
증상
· 반드시 끓인 물, 생수를 마시고 음식은 익혀먹기
· 올

In [30]:
# [셀8] 임베딩 모델 로딩 (첫 실행시 bge-m3 다운로드 ~2.3GB, 5~10분)
from langchain_community.embeddings import HuggingFaceEmbeddings

print("임베딩 모델 로딩 중...")
embed_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True, "batch_size": 32},
)
print("완료")

임베딩 모델 로딩 중...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 37362.12it/s]


완료


In [31]:
# [셀9] FAISS 인덱스 생성 + 저장
from langchain_community.vectorstores import FAISS

print(f"FAISS 인덱스 생성 중... ({len(chunks)}개 청크)")
vectordb = FAISS.from_documents(chunks, embed_model)
vectordb.save_local(str(INDEX_DIR))
print(f"저장 완료: {INDEX_DIR}")

FAISS 인덱스 생성 중... (305개 청크)
저장 완료: D:\rag_index


In [32]:
# [셀10] 검색 테스트
vectordb = FAISS.load_local(
    str(INDEX_DIR), embed_model,
    allow_dangerous_deserialization=True
)

queries = [
    "지진 발생 시 실내에서 가장 먼저 해야 할 행동은?",
    "화재 시 엘리베이터를 타도 되나요?",
    "태풍 경보 발령 기준 풍속은?",
    "홍수 예보 시 대피 순서는?",
    "미세먼지 나쁨 단계 기준 수치는?",
]

for q in queries:
    results = vectordb.similarity_search(q, k=2)
    print(f"\n[Q] {q}")
    for i, r in enumerate(results):
        print(f"  [{i+1}] {r.metadata['disaster_type']} | {r.metadata['source_file']}")
        print(f"       {r.page_content[:120]}...")


[Q] 지진 발생 시 실내에서 가장 먼저 해야 할 행동은?
  [1] 지진 | 미리 대비하고 알아두는 지진이야기_한페이지.pdf
       탁자 밑에 들어가 안전한 자세를 취한 후 진동이 멈추면 건물 밖으로 대피합니다. 
내진설계가 되어 있는 건물은 지진에 대한 안전성이 높을 수 있으니, 무리하게 대피하기 
보다는 진동이 멈추면 주변 상황을 살핀 후 대...
  [2] 지진 | 상황별 장소별 지진행동요령_3.pdf
       행정안전부 국민재난안전포털 safekorea gDkr 지진 발생 시 장소별 이렇게 행동하세요 학교 주택 백회점 마르 책상 아래로 들어가 책상다리틀 붙잡고 몸을 보호 흔들림이 멈추면 밖으로 대피 계단 기둥 근처예서 몸...

[Q] 화재 시 엘리베이터를 타도 되나요?
  [1] 화재 | (아파트 입주자) 화재 피난행동요령.pdf
       가스) 등이 빠른 속도로 상층부로 이동하므로 주의필요
●  화재 시 엘리베이터를 이용하는 경우 화재 시 뜨거워진 연기가 승강로를 상승하면서 엘리베이터 
내부로 침투하여 질식을 유발하거나, 정전 등의 이유로 정지하는 ...
  [2] 화재 | 250218_다중이용시설 화재 행동요령 카드뉴스.pdf
       소방청 행정안전부 국민행동요령 다중이용시설에서 화재 발생 시 이렇게 행동하세요
다중이용시설에서 소방청 화재 발생 시이렇게 행동하세요 행정안전부 시설이용자 엘레베이터 이용금지 대피 가능한 경우 젖은 수건 등올 활용해 ...

[Q] 태풍 경보 발령 기준 풍속은?
  [1] 강풍 | 강풍행동요령.docx
       강풍 발생으로 전력선이 차량에 닿는 경우, 차 안에 머무르면서 차의 금속 부분에 닿지 않도록 주의하고 주위 사람들에게 위험을 알리고 119에 연락하여 조치를 취하도록 합니다.
강풍
관련 정보
강풍에 대한 특보 기준을...
  [2] 지진 | 미리 대비하고 알아두는 지진이야기_한페이지.pdf
       지진속보
지진정보
국외 지진정보
발표
기준
규모 
5.0 이상
(우리나라의 지역) 규

In [33]:
# [셀11] 완료 요약
print("="*40)
print("  지식베이스 구축 완료")
print("="*40)
print(f"  문서 수  : {len(raw_docs)}개")
print(f"  청크 수  : {len(chunks)}개")
print(f"  임베딩   : BAAI/bge-m3")
print(f"  저장경로 : {INDEX_DIR}")
print()
print("다음: 02_qa_dataset.ipynb")

  지식베이스 구축 완료
  문서 수  : 44개
  청크 수  : 305개
  임베딩   : BAAI/bge-m3
  저장경로 : D:\rag_index

다음: 02_qa_dataset.ipynb
